# Repo Ingest Funnel Dashboard

Each cell below is a standalone SQL query designed to power one Lakeview dashboard widget.

### Key Tables & Date Columns
| Stage | Table | Date Column |
|-------|-------|-------------|
| Ingested | `openalex.repo.repo_works` | `updated_date` (OAI-PMH datestamp) |
| Scraped | `openalex.taxicab.taxicab_results` | `processed_date` (actual scrape time) |
| PDF parsed (GROBID) | `openalex.pdf.pdf_works` | `updated_date` (current_timestamp at parse) |
| Landing page parsed | `openalex.landing_page.landing_page_works` | `updated_date` (current_timestamp at parse) |
| PDF/OA enriched | `openalex.works.superlocations` | `updated_date` (source record date) |

## Query 1: Daily Funnel — Stage Counts by Day (7 days)

In [ ]:
WITH dates AS (
  SELECT explode(sequence(
    date_sub(current_date(), 7),
    date_sub(current_date(), 1),
    INTERVAL 1 DAY
  )) AS run_date
),
ingested AS (
  SELECT to_date(updated_date) AS dt,
    COUNT(*) AS records_ingested,
    SUM(CASE WHEN size(urls) > 0 THEN 1 ELSE 0 END) AS have_urls
  FROM openalex.repo.repo_works
  WHERE to_date(updated_date) >= date_sub(current_date(), 7)
  GROUP BY 1
),
scraped AS (
  SELECT to_date(processed_date) AS dt,
    COUNT(*) AS scraped,
    SUM(CASE WHEN http_status BETWEEN 200 AND 299 THEN 1 ELSE 0 END) AS scrape_success
  FROM openalex.taxicab.taxicab_results
  WHERE native_id_namespace = 'pmh'
    AND to_date(processed_date) >= date_sub(current_date(), 7)
  GROUP BY 1
),
pdf_parsed AS (
  SELECT to_date(updated_date) AS dt,
    COUNT(*) AS pdf_works_parsed
  FROM openalex.pdf.pdf_works
  WHERE to_date(updated_date) >= date_sub(current_date(), 7)
  GROUP BY 1
),
parsed AS (
  SELECT to_date(updated_date) AS dt,
    COUNT(*) AS landing_page_parsed,
    SUM(CASE WHEN size(pdf_urls) > 0 THEN 1 ELSE 0 END) AS pdf_links_found
  FROM openalex.landing_page.landing_page_works
  WHERE to_date(updated_date) >= date_sub(current_date(), 7)
  GROUP BY 1
),
enriched AS (
  SELECT to_date(updated_date) AS dt,
    SUM(CASE WHEN pdf_url IS NOT NULL THEN 1 ELSE 0 END) AS pdfs_retrieved,
    SUM(CASE WHEN is_oa = true THEN 1 ELSE 0 END) AS is_oa
  FROM openalex.works.superlocations
  WHERE to_date(updated_date) >= date_sub(current_date(), 7)
  GROUP BY 1
)
SELECT
  d.run_date,
  COALESCE(i.records_ingested, 0) AS records_ingested,
  COALESCE(i.have_urls, 0) AS have_urls,
  COALESCE(s.scraped, 0) AS scraped,
  COALESCE(s.scrape_success, 0) AS scrape_success,
  COALESCE(pdf.pdf_works_parsed, 0) AS pdf_works_parsed,
  COALESCE(p.landing_page_parsed, 0) AS landing_page_parsed,
  COALESCE(p.pdf_links_found, 0) AS pdf_links_found,
  COALESCE(e.pdfs_retrieved, 0) AS pdfs_retrieved,
  COALESCE(e.is_oa, 0) AS is_oa
FROM dates d
LEFT JOIN ingested i ON d.run_date = i.dt
LEFT JOIN scraped s ON d.run_date = s.dt
LEFT JOIN pdf_parsed pdf ON d.run_date = pdf.dt
LEFT JOIN parsed p ON d.run_date = p.dt
LEFT JOIN enriched e ON d.run_date = e.dt
ORDER BY d.run_date

## Query 2: 7-Day Funnel Summary

In [ ]:
SELECT
  (
    SELECT COUNT(*)
    FROM openalex.repo.repo_works
    WHERE to_date(updated_date) >= date_sub(current_date(), 7)
  ) AS records_ingested,
  (
    SELECT SUM(CASE WHEN size(urls) > 0 THEN 1 ELSE 0 END)
    FROM openalex.repo.repo_works
    WHERE to_date(updated_date) >= date_sub(current_date(), 7)
  ) AS have_urls,
  (
    SELECT COUNT(*)
    FROM openalex.taxicab.taxicab_results
    WHERE native_id_namespace = 'pmh'
      AND to_date(processed_date) >= date_sub(current_date(), 7)
  ) AS scraped,
  (
    SELECT SUM(CASE WHEN http_status BETWEEN 200 AND 299 THEN 1 ELSE 0 END)
    FROM openalex.taxicab.taxicab_results
    WHERE native_id_namespace = 'pmh'
      AND to_date(processed_date) >= date_sub(current_date(), 7)
  ) AS scrape_success,
  (
    SELECT COUNT(*)
    FROM openalex.pdf.pdf_works
    WHERE to_date(updated_date) >= date_sub(current_date(), 7)
  ) AS pdf_works_parsed,
  (
    SELECT COUNT(*)
    FROM openalex.landing_page.landing_page_works
    WHERE to_date(updated_date) >= date_sub(current_date(), 7)
  ) AS landing_page_parsed,
  (
    SELECT SUM(CASE WHEN size(pdf_urls) > 0 THEN 1 ELSE 0 END)
    FROM openalex.landing_page.landing_page_works
    WHERE to_date(updated_date) >= date_sub(current_date(), 7)
  ) AS pdf_links_found,
  (
    SELECT SUM(CASE WHEN pdf_url IS NOT NULL THEN 1 ELSE 0 END)
    FROM openalex.works.superlocations
    WHERE to_date(updated_date) >= date_sub(current_date(), 7)
  ) AS pdfs_retrieved,
  (
    SELECT SUM(CASE WHEN is_oa = true THEN 1 ELSE 0 END)
    FROM openalex.works.superlocations
    WHERE to_date(updated_date) >= date_sub(current_date(), 7)
  ) AS is_oa

## Query 3: Taxicab Scrape Outcomes by Day

In [ ]:
SELECT
  to_date(processed_date) AS run_date,
  COUNT(*) AS total_scraped,
  SUM(CASE WHEN http_status BETWEEN 200 AND 299 THEN 1 ELSE 0 END) AS success,
  SUM(CASE WHEN http_status >= 400 OR http_status IS NULL THEN 1 ELSE 0 END) AS errors,
  SUM(CASE WHEN http_status = 403 OR http_status = 429 THEN 1 ELSE 0 END) AS soft_blocks,
  ROUND(
    100.0 * SUM(CASE WHEN http_status BETWEEN 200 AND 299 THEN 1 ELSE 0 END) / COUNT(*),
    1
  ) AS success_rate_pct
FROM openalex.taxicab.taxicab_results
WHERE native_id_namespace = 'pmh'
  AND to_date(processed_date) >= date_sub(current_date(), 7)
GROUP BY 1
ORDER BY 1

## Query 4: Cohort Tracker

Follow records ingested on each day through downstream stages.
Downstream checks are existence-based (not date-based) since pipeline lag means today's records won't reach superlocations until tonight's E2E.

pdf_works join: repo_works.native_id matches pdf_works.ids where namespace = 'pmh'.

**Note**: This query joins large tables — run on x-large warehouse if slow.

In [ ]:
WITH cohort AS (
  SELECT
    to_date(updated_date) AS ingest_date,
    native_id,
    CASE WHEN size(urls) > 0 THEN 1 ELSE 0 END AS has_url
  FROM openalex.repo.repo_works
  WHERE to_date(updated_date) >= date_sub(current_date(), 7)
),
pdf_pmh_ids AS (
  SELECT DISTINCT id_entry.id AS pmh_id
  FROM openalex.pdf.pdf_works
  LATERAL VIEW explode(ids) AS id_entry
  WHERE id_entry.namespace = 'pmh'
)
SELECT
  c.ingest_date,
  COUNT(*) AS records_ingested,
  SUM(c.has_url) AS have_urls,
  SUM(CASE WHEN t.native_id IS NOT NULL THEN 1 ELSE 0 END) AS scraped,
  SUM(CASE WHEN t.http_status BETWEEN 200 AND 299 THEN 1 ELSE 0 END) AS scrape_success,
  SUM(CASE WHEN pdf.pmh_id IS NOT NULL THEN 1 ELSE 0 END) AS pdf_parsed,
  SUM(CASE WHEN sl.native_id IS NOT NULL THEN 1 ELSE 0 END) AS in_superlocations
FROM cohort c
LEFT JOIN openalex.taxicab.taxicab_results t
  ON c.native_id = t.native_id AND t.native_id_namespace = 'pmh'
LEFT JOIN pdf_pmh_ids pdf
  ON c.native_id = pdf.pmh_id
LEFT JOIN openalex.works.superlocations sl
  ON c.native_id = sl.native_id
GROUP BY c.ingest_date
ORDER BY c.ingest_date

## Query 5: Top Repositories by Volume (7 days)

In [ ]:
SELECT
  repository_id,
  COUNT(*) AS records_ingested,
  SUM(CASE WHEN size(urls) > 0 THEN 1 ELSE 0 END) AS have_urls,
  ROUND(
    100.0 * SUM(CASE WHEN size(urls) > 0 THEN 1 ELSE 0 END) / COUNT(*),
    1
  ) AS pct_with_urls
FROM openalex.repo.repo_works
WHERE to_date(updated_date) >= date_sub(current_date(), 7)
GROUP BY repository_id
ORDER BY records_ingested DESC
LIMIT 25

## Query 6: URL-less Records by Day

In [ ]:
SELECT
  to_date(updated_date) AS run_date,
  COUNT(*) AS total_records,
  SUM(CASE WHEN urls IS NULL OR size(urls) = 0 THEN 1 ELSE 0 END) AS no_url_records,
  SUM(CASE WHEN size(urls) > 0 THEN 1 ELSE 0 END) AS have_url_records,
  ROUND(
    100.0 * SUM(CASE WHEN urls IS NULL OR size(urls) = 0 THEN 1 ELSE 0 END) / COUNT(*),
    1
  ) AS pct_no_url
FROM openalex.repo.repo_works
WHERE to_date(updated_date) >= date_sub(current_date(), 7)
GROUP BY 1
ORDER BY 1